# Tornado Project — Milestone 3: Preprocessing
**Target:** `HIGH_PROPERTY_DAMAGE` (reported property damage ≥ $100,000).  
**Prediction moment:** at tornado onset, **not after the tornado ends**.  
**Input:** `tornado_m1_cleaned.xlsx` in the same folder. Run cells from top to bottom.   
**Professor feedback addressed:** prediction-time availability, redundant ending coordinates and length/duration, zero-duration records, plot-to-decision links, and time/geographic generalization.

## 1. Split your data
**Decision:** stratified random 80/20 train/test split, seed 42, to preserve the minority high-damage class

**Prediction-time feature policy:** keep `YEAR`, `MONTH_NAME`, `BEGIN_HOUR`, `STATE`, and provisionally `WFO`, `BEGIN_LAT`, `BEGIN_LON`. Exclude `PROPERTY_DAMAGE_USD`, `TOR_LENGTH`, `TOR_WIDTH`, `DURATION_MINUTES`, `END_LAT`, `END_LON`, and any post-event attributes. `BEGIN_RANGE`/`BEGIN_AZIMUTH` are excluded until availability is documented.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("tornado_m1_cleaned(2).xlsx")
raw = pd.read_excel(DATA_PATH)
TARGET = "HIGH_PROPERTY_DAMAGE"
FEATURES = ["YEAR", "MONTH_NAME", "BEGIN_HOUR", "STATE", "WFO", "BEGIN_LAT", "BEGIN_LON"]
assert set(FEATURES + [TARGET]).issubset(raw.columns), set(FEATURES + [TARGET]) - set(raw.columns)
# Remove only completely empty spreadsheet rows; no fitted transformation.
df = raw.dropna(how="all").copy()
assert df[TARGET].notna().all()
assert set(df[TARGET].unique()).issubset({0, 1})
X = df[FEATURES].copy()
y = df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Full:", X.shape, "Train:", X_train.shape, "Test:", X_test.shape)
print("Class fractions (train/test):", y_train.mean(), y_test.mean())
assert X_train.index.intersection(X_test.index).empty


Full: (21892, 7) Train: (17513, 7) Test: (4379, 7)
Class fractions (train/test): 0.24130645805972706 0.2413793103448276


We split the dataset 80% training and 20% testing with random_state=42, stratified by HIGH_PROPERTY_DAMAGE. All preprocessing estimators will be fitted on the training portion only. The test portion will be transformed using those fitted estimators. **17,513 training rows at 24.13% high damage** and **4,379 test rows at 24.14%**.

## 2. Handle missing values
**M2 finding:** valid records appeared to have no missing values. Even if counts are zero, use a pipeline imputer to handle future missing numeric/categorical inputs.

In [2]:
print("Training missing values by column:")
print(X_train.isna().sum().to_string())
print("Test missing values:")
print(X_test.isna().sum().to_string())


Training missing values by column:
YEAR          0
MONTH_NAME    0
BEGIN_HOUR    0
STATE         0
WFO           0
BEGIN_LAT     0
BEGIN_LON     0
Test missing values:
YEAR          0
MONTH_NAME    0
BEGIN_HOUR    0
STATE         0
WFO           0
BEGIN_LAT     0
BEGIN_LON     0


After splitting the dataset, we checked the training data for missing values. Based on my Milestone 2 findings, the cleaned dataset contained no missing values, so no actual imputation was necessary. However, we included an imputer in the preprocessing pipeline to handle any missing values that might appear in future data. The imputer is fitted only on the training data to prevent data leakage.

We also decided not to create missingness indicators because there were no missing values in the cleaned dataset, and my M2 analysis did not identify a relationship between missingness and high property damage.

In [3]:
print("Training-only numeric summary:")
print(X_train.select_dtypes(include="number").describe().T)
print("Invalid training coordinate counts:")
print("Latitude:", (~X_train["BEGIN_LAT"].between(-90, 90) & X_train["BEGIN_LAT"].notna()).sum())
print("Longitude:", (~X_train["BEGIN_LON"].between(-180, 180) & X_train["BEGIN_LON"].notna()).sum())
# Audit excluded fields only on the TRAINING ROWS; do not use them to select/filter examples.
if "DURATION_MINUTES" in df.columns:
    duration_train = df.loc[X_train.index, "DURATION_MINUTES"]
    print("Training zero-duration records:", (duration_train == 0).sum())
if {"TOR_LENGTH", "TOR_WIDTH"}.issubset(df.columns):
    print(df.loc[X_train.index, ["TOR_LENGTH", "TOR_WIDTH"]].describe().T)


Training-only numeric summary:
              count         mean       std        min       25%        50%  \
YEAR        17513.0  2015.354993  5.349257  2007.0000  2011.000  2015.0000   
BEGIN_HOUR  17513.0    14.803346  5.333685     0.0000    13.000    16.0000   
BEGIN_LAT   17513.0    36.974209  4.605823    17.7212    33.235    36.6414   
BEGIN_LON   17513.0   -91.681032  7.875884  -159.6577   -96.900   -91.4952   

                  75%        max  
YEAR        2020.0000  2024.0000  
BEGIN_HOUR    18.0000    23.0000  
BEGIN_LAT     40.5408    61.1300  
BEGIN_LON    -86.5234   -64.7151  
Invalid training coordinate counts:
Latitude: 0
Longitude: 0
Training zero-duration records: 931
              count        mean         std   min    25%   50%     75%  \
TOR_LENGTH  17513.0    3.157669    4.245207  0.01   0.49   1.6    4.12   
TOR_WIDTH   17513.0  189.770616  294.795367  1.00  50.00  80.0  200.00   

                max  
TOR_LENGTH    49.96  
TOR_WIDTH   4576.00  


After examining the training data, we found 931 tornado records with a duration of zero minutes. These could represent very brief tornadoes, rounded timestamps, or recording errors. Since the model predicts property damage at the beginning of a tornado, we decided to keep these records because the final duration would not be known at prediction time.

We also checked the starting coordinates for unusual values. All latitude and longitude values were within valid geographic ranges, with no invalid or zero-valued coordinates. Because of this we decided not to remove or modify any records based on their coordinates. We will retain extreme observations unless there is clear evidence of a recording error.

## 4. Encode categorical features
We will use one-hot encoding for `STATE`, `WFO`, `MONTH_NAME` and engineered `TORNADO_SEASON`. Their labels do not have a meaningful numerical order. Encoder fitting occurs on training data inside the pipeline.

We chose one-hot encoding for these variable because these categories do not have a meaningful numerical order. One-hot encoding converts each category into separate numerical columns so the model can process them without assuming one category is greater than another. We also used handle_unknown="ignore" so the model can handle categories in the test data that were not present during training. The encoder is fitted only on the training data within the preprocessing pipeline to prevent data leakage.

## 5. Scale numerical features
We chose StandardScaler to scale the numerical features in the tornado dataset, including YEAR, BEGIN_HOUR, BEGIN_LAT, and BEGIN_LON, along with the numerical features created during feature engineering.

StandardScaler transforms numerical values so they have a mean of approximately 0 and a standard deviation of 1. This prevents features with larger numerical ranges from having too much influence on models that are sensitive to feature scale. We included StandardScaler inside the preprocessing pipeline so it learns the mean and standard deviation from the training data only. This prevents information from the test data from influencing the scaling process.
Scaling is especially useful for Logistic Regression, SVM, and KNN because they are sensitive to feature scales. Decision Trees and Random Forests generally do not need scaling. We kept StandardScaler in the pipeline so we can compare different models in M4 using the same preprocessing steps.


## 6. Engineer at least three domain-based features
1. **Season:** derive meteorological season from month, reflecting seasonal tornado conditions.
2. **Hour cycle:** derive `HOUR_SIN` and `HOUR_COS` so 23:00 and 00:00 are close, unlike raw clock integers.
3. **Geographic interaction:** `LAT_LON_PRODUCT` combines starting latitude and longitude to give the model a possible interaction between locations. This is an experimental geographic feature, not a measure of tornado strength, so we will check whether it actually helps in M4.

All use onset-time inputs. These are derived variables, not renames. The transformer below is deterministic and stateless; it runs *inside* the pipeline after splitting.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class TornadoFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self  # no statistics are learned

    def transform(self, X):
        X = X.copy()
        month_lookup = {"JANUARY":1,"FEBRUARY":2,"MARCH":3,"APRIL":4,"MAY":5,"JUNE":6,
                        "JULY":7,"AUGUST":8,"SEPTEMBER":9,"OCTOBER":10,"NOVEMBER":11,"DECEMBER":12}
        month_num = X["MONTH_NAME"].astype("string").str.upper().str.strip().map(month_lookup)
        # Missing/unrecognized months remain missing; categorical imputer handles them.
        season_map = {12:"Winter",1:"Winter",2:"Winter",3:"Spring",4:"Spring",5:"Spring",
                      6:"Summer",7:"Summer",8:"Summer",9:"Fall",10:"Fall",11:"Fall"}
        X["TORNADO_SEASON"] = month_num.map(season_map).astype(object)
        hour = pd.to_numeric(X["BEGIN_HOUR"], errors="coerce")
        X["HOUR_SIN"] = np.sin(2 * np.pi * hour / 24)
        X["HOUR_COS"] = np.cos(2 * np.pi * hour / 24)
        lat = pd.to_numeric(X["BEGIN_LAT"], errors="coerce")
        lon = pd.to_numeric(X["BEGIN_LON"], errors="coerce")
        X["LAT_LON_PRODUCT"] = lat * lon
        return X

sample_engineered = TornadoFeatureEngineer().transform(X_train.head(5))
print(sample_engineered[["TORNADO_SEASON", "HOUR_SIN", "HOUR_COS", "LAT_LON_PRODUCT"]])


      TORNADO_SEASON  HOUR_SIN  HOUR_COS  LAT_LON_PRODUCT
9715          Summer -0.500000 -0.866025     -3256.633142
265           Spring -0.866025 -0.500000     -3484.769100
12239         Winter -0.965926 -0.258819     -3656.004630
19630         Spring  0.258819 -0.965926     -3274.139500
17598         Summer -0.866025 -0.500000     -3086.885313


## 7. Build a Pipeline / ColumnTransformer
The feature engineer creates new columns; `ColumnTransformer` then imputes, scales, and one-hot encodes. `fit_transform` is called **only on training data**; `transform` is called on test data. `handle_unknown="ignore"` prevents unseen-category errors.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

raw_numeric_features = ["YEAR", "BEGIN_HOUR", "BEGIN_LAT", "BEGIN_LON"]
raw_categorical_features = ["MONTH_NAME", "STATE", "WFO"]
numeric_features = raw_numeric_features + ["HOUR_SIN", "HOUR_COS", "LAT_LON_PRODUCT"]
categorical_features = raw_categorical_features + ["TORNADO_SEASON"]

# Fill missing inputs before deriving features from them.
raw_imputer = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), raw_numeric_features),
    ("categorical", SimpleImputer(strategy="most_frequent"), raw_categorical_features),
], verbose_feature_names_out=False).set_output(transform="pandas")

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
full_pipeline = Pipeline([
    ("impute", raw_imputer),
    ("engineer", TornadoFeatureEngineer()),
    ("preprocess", preprocessor),
])

X_train_processed = full_pipeline.fit_transform(X_train, y_train)
X_test_processed = full_pipeline.transform(X_test)
print("Training transformed:", X_train_processed.shape)
print("Testing transformed:", X_test_processed.shape)


Training transformed: (17513, 194)
Testing transformed: (4379, 194)


## 8. Leakage audit and validation plan
**Target leakage:** `PROPERTY_DAMAGE_USD` directly defines `HIGH_PROPERTY_DAMAGE`; it is never included as feature. To preserve future data leakage, final length, width, duration, ending coordinates, and post-event reports are also excluded. We split before fitting. The engineered features do not learn dataset statistics.

**Prediction-time availability:** Our model is intended to predict high property damage at tornado onset. We excluded final tornado measurements and the reported property damage amount because these are not known at that time. We are keeping `BEGIN_LAT`, `BEGIN_LON`, `WFO`, and `STATE` as provisional features, but we still need to verify that their values would be available at the moment of prediction. If the starting coordinates were finalized only after the tornado, we would remove them and `LAT_LON_PRODUCT`. Until this is checked, our notebook is a baseline rather than a verified real-time model.

**Generalization plan (M4):** compare the random stratified split (train on earlier years, evaluate on later years), and optionally hold out states or geographic groups. Refit the entire pipeline inside each training partition; never reuse preprocessing fitted on a different split. Report differences in F1-score. Do not mix geographic or chronological validation into the random split without clearly distinguishing the experiments.

In [6]:
# Structural checks of leakage prevention
for forbidden in ["PROPERTY_DAMAGE_USD", "TOR_LENGTH", "TOR_WIDTH", "DURATION_MINUTES", "END_LAT", "END_LON"]:
    assert forbidden not in X_train.columns
assert list(X_train.columns) == FEATURES
assert X_train.index.intersection(X_test.index).empty
print("Leakage structural checks passed. Prediction-time provenance still requires verification.")

# Check that a WFO absent from training keeps the same feature matrix width.
unseen_wfo = X_train.iloc[[0]].assign(WFO="UNSEEN_WFO")
transformed = full_pipeline.transform(unseen_wfo)
feature_names = full_pipeline.named_steps["preprocess"].get_feature_names_out()
wfo_columns = [name.startswith("categorical__WFO_") for name in feature_names]
assert any(wfo_columns)
assert transformed.shape == (1, len(feature_names))
wfo_values = transformed[:, wfo_columns]
assert np.all((wfo_values.toarray() if hasattr(wfo_values, "toarray") else wfo_values) == 0)
print("Unseen WFO check passed.")


Leakage structural checks passed. Prediction-time provenance still requires verification.
Unseen WFO check passed.


## 9. Show before and after
Compare shapes, missingness, sample columns, and finite transformed numeric values. These outputs make the transformations visible and provide evidence for the checkpoint.

In [7]:
print("BEFORE — training features:", X_train.shape)
print("BEFORE — test features:", X_test.shape)
print("BEFORE — missing cells (training):", int(X_train.isna().sum().sum()))
print("BEFORE — training sample:")
print(X_train.head(3).to_string())

feature_names = full_pipeline.named_steps["preprocess"].get_feature_names_out()
print("AFTER — training matrix:", X_train_processed.shape)
print("AFTER — test matrix:", X_test_processed.shape)
print("AFTER — feature count:", len(feature_names))
print("AFTER — first 25 feature names:", feature_names[:25].tolist())
print("AFTER — training sample (first 12 features):")
preview = X_train_processed[:3, :12]
print(preview.toarray() if hasattr(preview, "toarray") else preview)
assert X_train_processed.shape[0] == len(X_train)
assert X_test_processed.shape[0] == len(X_test)
assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("BEFORE — training numeric summary:")
print(X_train[raw_numeric_features].describe().T.round(3).to_string())

scaled = X_train_processed[:, :len(numeric_features)]
scaled = scaled.toarray() if hasattr(scaled, "toarray") else scaled
print("AFTER — scaled training numeric summary:")
print(pd.DataFrame(scaled, columns=numeric_features).describe().T.round(3).to_string())

for name, matrix in [("training", X_train_processed), ("test", X_test_processed)]:
    values = matrix.data if hasattr(matrix, "toarray") else matrix
    nonfinite = np.count_nonzero(~np.isfinite(values))
    print(f"AFTER — non-finite cells ({name}):", nonfinite)
    assert nonfinite == 0


BEFORE — training features: (17513, 7)
BEFORE — test features: (4379, 7)
BEFORE — missing cells (training): 0
BEFORE — training sample:
       YEAR MONTH_NAME  BEGIN_HOUR      STATE  WFO  BEGIN_LAT  BEGIN_LON
9715   2014       June          14  TENNESSEE  MEG    36.4445   -89.3587
265    2007      March          16   MISSOURI  LSX    38.6124   -90.2500
12239  2017   February          17   ILLINOIS  ILX    40.9128   -89.3609
AFTER — training matrix: (17513, 194)
AFTER — test matrix: (4379, 194)
AFTER — feature count: 194
AFTER — first 25 feature names: ['numeric__YEAR', 'numeric__BEGIN_HOUR', 'numeric__BEGIN_LAT', 'numeric__BEGIN_LON', 'numeric__HOUR_SIN', 'numeric__HOUR_COS', 'numeric__LAT_LON_PRODUCT', 'categorical__MONTH_NAME_April', 'categorical__MONTH_NAME_August', 'categorical__MONTH_NAME_December', 'categorical__MONTH_NAME_February', 'categorical__MONTH_NAME_January', 'categorical__MONTH_NAME_July', 'categorical__MONTH_NAME_June', 'categorical__MONTH_NAME_March', 'categorical__MO

### Final checkpoint interpretation
- **Split:** 17,513 training rows and 4,379 test rows. The high-damage shares were 24.13% and 24.14%, respectively.
- **Missingness:** 0 missing training cells. For future missing values, training medians limit the influence of numeric extremes and the most frequent training labels provide valid categorical fills.
- **Outliers:** 931 zero-duration training records were kept because final duration is unavailable at onset. The training data had 0 out-of-range starting coordinates, so no clipping was needed.
- **Encoding/scaling:** 187 one-hot indicators encode categories without imposing a ranking. The 7 standardized numeric features are on comparable scales for regularized logistic regression. Random forest splits depend on feature order, which standardization preserves.
- **Engineering:** Season represents broad seasonal tornado conditions. Cyclical hour keeps adjacent hours close at midnight. The geographic interaction gives a linear model a way to consider latitude and longitude together, although it is not a direct measure of tornado severity. Its predictive value and the coordinates’ availability at onset still need verification.
- **Leakage:** exclude target-derived and final-event fields; preprocessing fit on training only.
- **Generalization:** later-year and geographic holdout proposed for M4.
- **Before/after:** Both folds went from 7 to 194 columns and kept all their rows. The transformed matrices contained 0 non-finite values. Scaled training features have means near zero and standard deviations near one.

**Figure-to-decision bridge from M2:** length/duration correlation → exclude both as post-event; beginning/ending coordinates correlation → retain only onset-available start location; extreme duration/path distributions → audit but do not use to filter onset training data; target imbalance → stratify and use F1 in M4.
